# Bronze Ingestion

This notebook ingests all eight TPC-H source tables into the Bronze layer.

Bronze preserves source data without transformation. Only ingestion metadata is added:

- `_ingested_at`
- `_source_table`

Bronze tables are stored in `workspace.default` using the `finance_bronze_` table-name prefix.

## 1. Configuration

Project naming is centralized in `src.config` so that table references remain consistent across the repository.

In [0]:
from src.config import (
    TABLES,
    BRONZE_PREFIX,
    source_table,
    project_table,
)

from src.synevyr_finance_lakehouse.bronze import ingest_to_bronze

print("Bronze targets:")
for table in TABLES:
    print(f"{source_table(table)} -> {project_table(BRONZE_PREFIX, table)}")

## 2. Bronze Ingestion

Each source table is copied into a Delta table in `workspace.default`.

No source columns are renamed, filtered, cleaned, or otherwise transformed.

In [0]:
for table in TABLES:
    source_name = source_table(table)
    target_name = project_table(BRONZE_PREFIX, table)

    print(f"Ingesting {source_name} -> {target_name}")

    ingest_to_bronze(
        spark=spark,
        source_table_name=source_name,
        target_table_name=target_name,
    )

print("Bronze ingestion complete.")

## 3. Table Existence Check

Verify that all expected Bronze tables were created successfully.

In [0]:
display(
    spark.sql(
        "SHOW TABLES IN workspace.default LIKE 'finance_bronze_*'"
    )
)

## 4. Row Count Validation

Source and Bronze row counts are compared to confirm that ingestion did not silently drop or duplicate records.

In [0]:
row_count_checks = []

for table in TABLES:
    source_name = source_table(table)
    bronze_name = project_table(BRONZE_PREFIX, table)

    source_count = spark.table(source_name).count()
    bronze_count = spark.table(bronze_name).count()

    row_count_checks.append(
        (
            table,
            source_count,
            bronze_count,
            source_count - bronze_count,
        )
    )

row_count_checks_df = spark.createDataFrame(
    row_count_checks,
    [
        "table_name",
        "source_count",
        "bronze_count",
        "difference",
    ],
)

display(
    row_count_checks_df.orderBy("table_name")
)

### Expected result

`difference` should be `0` for every table.

## 5. Schema Preservation Validation

Bronze is expected to contain all original source columns plus exactly two metadata columns:

- `_ingested_at`
- `_source_table`

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    ArrayType,
)

METADATA_COLUMNS = {
    "_ingested_at",
    "_source_table",
}

source_dfs = {
    table: spark.table(source_table(table))
    for table in TABLES
}

bronze_dfs = {
    table: spark.table(project_table(BRONZE_PREFIX, table))
    for table in TABLES
}

source_schemas = {
    table: df.schema
    for table, df in source_dfs.items()
}

bronze_schemas = {
    table: df.schema
    for table, df in bronze_dfs.items()
}

schema_checks = []

for table in TABLES:
    source_columns = set(source_schemas[table].names)
    bronze_columns = set(bronze_schemas[table].names)

    expected_columns = source_columns | METADATA_COLUMNS

    schema_checks.append(
        (
            table,
            bronze_columns == expected_columns,
            sorted(expected_columns - bronze_columns),
            sorted(bronze_columns - expected_columns),
        )
    )

schema_checks_schema = StructType([
    StructField("table_name", StringType(), False),
    StructField("schema_matches", BooleanType(), False),
    StructField("missing_columns", ArrayType(StringType()), False),
    StructField("unexpected_columns", ArrayType(StringType()), False),
])

schema_checks_df = spark.createDataFrame(
    schema_checks,
    schema=schema_checks_schema,
)

display(schema_checks_df)

### Expected result

`schema_matches` should be `true` for every table, with no missing or unexpected columns.

## 6. Metadata Spot Check

Inspect a small sample to confirm that ingestion metadata was added correctly.

In [0]:
display(
    spark.table(
        project_table(BRONZE_PREFIX, "customer")
    )
    .select(
        "c_custkey",
        "_source_table",
        "_ingested_at",
    )
    .limit(10)
)

## 7. Bronze Ingestion Conclusion

If all checks above pass:

- all eight TPC-H source tables were successfully ingested into Bronze as Delta tables;
- original source columns were preserved unchanged;
- only `_ingested_at` and `_source_table` were added;
- source and Bronze row counts match;
- no source columns were dropped or unexpectedly added.

The Bronze layer is therefore ready to serve as the raw-data foundation for Silver transformations.